# 10 · Challengers tabulares

**Objetivo.** Saber si XGBoost, LightGBM o Extra Trees predicen mejor que el `HistGradientBoostingRegressor` actual sin cambiar simultáneamente los datos ni la evaluación.

**Baseline del MAE.** Participación anterior de la misma familia electoral para sección estable o componente oficial A+B (`lag_same_type_stable_plus_official_lineage`).

**Control experimental.** Todos usan `territorial_context`, folds 2018/2021/2024, pesos por registros identificados, predicción residual con contracción 0.25 y la misma corrección nacional. Cada familia tiene una especificación moderada congelada; no hay búsqueda masiva con sólo tres targets.

**Gate.** Un challenger debe mejorar el MAE ponderado 2021 frente al boosting actual y no deteriorar 2018 ni 2024 más de 0.5 pp.

In [ ]:
from pathlib import Path
import json
import time

import pandas as pd
import yaml
from IPython.display import Markdown, display

from radar27.modeling.tabular_challengers import (
    run_tabular_challengers,
    write_tabular_challenger_decision,
)


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (
            candidate / "configs/tabular_challengers.yaml"
        ).exists():
            return candidate
    raise FileNotFoundError("No se encontró la raíz del proyecto")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
RUN_STARTED = time.perf_counter()
with (PROJECT_ROOT / "configs/tabular_challengers.yaml").open(encoding="utf-8") as stream:
    challenger_config = yaml.safe_load(stream)
mart_path = PROJECT_ROOT / "artifacts/tables/feature_ablation_mart.parquet"
assert mart_path.exists()
challenger_mart = pd.read_parquet(mart_path)
display(pd.DataFrame(challenger_config["models"]).T)
display(Markdown(
    f"Se comparan `{len(challenger_config['models'])}` familias sobre "
    f"`{len(challenger_mart):,}` filas del mart y las mismas variables territoriales."
))

## Backtests fuera de tiempo

Para cada elección, el entrenamiento contiene únicamente elecciones anteriores. Cada algoritmo ajusta un modelo directo de respaldo y otro que estima cuánto corregir el baseline.

In [ ]:
challenger_predictions, challenger_metrics, challenger_registry = (
    run_tabular_challengers(PROJECT_ROOT, challenger_mart)
)
metric_view = challenger_metrics.copy()
for column in [
    "baseline_weighted_mae", "identified_weighted_mae",
    "mae_change_vs_baseline", "mae_change_vs_reference",
]:
    metric_view[column] = 100 * metric_view[column]
display(metric_view[[
    "target_year", "model_id", "baseline_weighted_mae",
    "identified_weighted_mae", "mae_change_vs_reference", "fit_seconds",
]])

In [ ]:
audit_checks = {
    "same_feature_group": set(challenger_metrics.feature_group) == {"territorial_context"},
    "all_models_all_folds": all(
        set(group.target_year) == {2018, 2021, 2024}
        for _, group in challenger_metrics.groupby("model_id")
    ),
    "training_strictly_prior": all(
        all(int(year) < int(row.target_year) for year in str(row.training_target_years).split("|"))
        for row in challenger_registry.itertuples(index=False)
    ),
    "same_numeric_features_per_fold": bool(
        challenger_registry.groupby("target_year").numeric_features.nunique().eq(1).all()
    ),
    "same_categorical_features_per_fold": bool(
        challenger_registry.groupby("target_year").categorical_features.nunique().eq(1).all()
    ),
}
display(pd.Series(audit_checks, name="aprobado").to_frame())
assert all(audit_checks.values())

## Decisión

2021 es el target principal congelado. Los otros dos años protegen contra una mejora accidental de un solo ciclo.

In [ ]:
challenger_decision = write_tabular_challenger_decision(
    PROJECT_ROOT, challenger_metrics
)
display(pd.Series(challenger_decision).to_frame("valor"))

In [ ]:
report_path = PROJECT_ROOT / "artifacts/reports/tabular_challengers.html"
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text("\n".join([
    "<h1>RADAR 27 · Challengers tabulares</h1>",
    "<p>Generado por notebooks/10_tabular_challengers.ipynb.</p>",
    "<h2>Métricas</h2>", challenger_metrics.to_html(index=False),
    "<h2>Registro</h2>", challenger_registry.to_html(index=False),
    "<h2>Decisión</h2><pre>" + json.dumps(
        challenger_decision, ensure_ascii=False, indent=2
    ) + "</pre>",
]), encoding="utf-8")

required_outputs = [
    PROJECT_ROOT / "artifacts/tables/tabular_challenger_predictions.parquet",
    PROJECT_ROOT / "artifacts/tables/tabular_challenger_metrics.csv",
    PROJECT_ROOT / "artifacts/tables/tabular_challenger_registry.csv",
    PROJECT_ROOT / "artifacts/tabular_challenger_decision.json",
    report_path,
]
final_checks = {
    "all_outputs_exist": all(path.exists() for path in required_outputs),
    **audit_checks,
}
display(pd.Series(final_checks, name="aprobado").to_frame())
assert all(final_checks.values())

In [ ]:
display(Markdown(
    f"**Cierre.** Mejor modelo en 2021: `{challenger_decision['best_primary_model']}`; "
    f"challenger promovido: `{challenger_decision['challenger_promoted']}`; "
    f"selección: `{challenger_decision['selected_model_for_next_iteration']}`. "
    f"Tiempo total: `{time.perf_counter()-RUN_STARTED:.1f}` s."
))